In [1]:
import pandas as pd

df2 = pd.read_csv('../csv/processed/C123_scaled_master_table.csv')

df2.describe

<bound method NDFrame.describe of          hs6  year       hs6_name_kr cbam_category  kr_eu_export_weight_kg  \
0     252310  2020  시멘트 클링커(clinker)           시멘트                    74.0   
1     252310  2022  시멘트 클링커(clinker)           시멘트                     0.0   
2     252310  2023  시멘트 클링커(clinker)           시멘트                     0.0   
3     252310  2024  시멘트 클링커(clinker)           시멘트                     0.0   
4     252310  2025  시멘트 클링커(clinker)           시멘트                     0.0   
...      ...   ...               ...           ...                     ...   
1233  761699  2021         알루미늄 기타제품          알루미늄               2982718.0   
1234  761699  2022         알루미늄 기타제품          알루미늄               3226679.0   
1235  761699  2023         알루미늄 기타제품          알루미늄               2630022.0   
1236  761699  2024         알루미늄 기타제품          알루미늄               2144975.0   
1237  761699  2025         알루미늄 기타제품          알루미늄               1783760.0   

      kr_eu_export_value_usd 

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings

warnings.filterwarnings('ignore')

# 1. 데이터 불러오기 및 전처리
print("데이터를 불러오는 중입니다...")
df = pd.read_csv('../csv/processed/C123_scaled_master_table.csv')

# 2026년 예측을 위해 2018~2025년 데이터만 학습에 사용
df_train = df[df['year'] < 2026].copy()

# 산출된 가중치 설정
weights = {'c1': 0.3986, 'c2': 0.3525, 'c3': 0.2489}
features = ['c1_scaled', 'c2_scaled', 'c3_scaled']

# 클리핑(Clipping) 범위 설정 (정규분포 기준 -5.2 ~ 5.2)
CLIP_MIN, CLIP_MAX = -5.2, 5.2

# 결과 저장을 위한 리스트
results_lr = [] # 선형 회귀 결과
results_es = [] # 지수평활법 결과

hs6_list = df_train['hs6'].unique()

print("[2026년 리스크 지표 예측 중 (선형회귀 & 지수평활법)]...")

for hs in hs6_list:
    item_data = df_train[df_train['hs6'] == hs].sort_values('year')
    
    # 데이터가 3년 치 이상일 때만 추세 예측
    if len(item_data) >= 3:
        X_years = item_data[['year']].values
        
        pred_lr_dict = {'hs6': hs, 'year': 2026}
        pred_es_dict = {'hs6': hs, 'year': 2026}
        
        for col in features:
            y_values = item_data[col].values
            
            # ----------------------------------------------------
            # 모델 1: 선형 회귀 (Linear Regression)
            # ----------------------------------------------------
            model_lr = LinearRegression()
            model_lr.fit(X_years, y_values)
            pred_lr = model_lr.predict([[2026]])[0]
            
            # ----------------------------------------------------
            # 모델 2: 지수평활법 (Holt's Linear Trend)
            # ----------------------------------------------------
            # 추세(trend)가 있는 지수평활법 적용. 
            # 데이터가 일직선이거나 분산이 없어 에러가 날 경우 선형회귀 값으로 대체
            try:
                model_es = ExponentialSmoothing(y_values, trend='add', initialization_method='estimated')
                fit_es = model_es.fit()
                pred_es = fit_es.forecast(1)[0]
            except:
                pred_es = pred_lr
            
            # --- 예측값 Clipping (-5.2 ~ 5.2 제한) ---
            pred_lr_dict[f'{col}_pred'] = np.clip(pred_lr, CLIP_MIN, CLIP_MAX)
            pred_es_dict[f'{col}_pred'] = np.clip(pred_es, CLIP_MIN, CLIP_MAX)
            
        results_lr.append(pred_lr_dict)
        results_es.append(pred_es_dict)

# 데이터프레임 변환
df_pred_lr = pd.DataFrame(results_lr)
df_pred_es = pd.DataFrame(results_es)

# ----------------------------------------------------
# 2. Risk Score 산출 및 정렬
# ----------------------------------------------------
# 가중치를 적용하여 최종 Risk Score 계산 (C1*39.86% + C2*35.25% + C3*24.89%)
for df_pred in [df_pred_lr, df_pred_es]:
    df_pred['Risk_Score'] = (
        df_pred['c1_scaled_pred'] * weights['c1'] + 
        df_pred['c2_scaled_pred'] * weights['c2'] + 
        df_pred['c3_scaled_pred'] * weights['c3']
    )

# Risk Score 기준으로 내림차순 정렬
df_pred_lr = df_pred_lr.sort_values(by='Risk_Score', ascending=False).reset_index(drop=True)
df_pred_es = df_pred_es.sort_values(by='Risk_Score', ascending=False).reset_index(drop=True)

# ----------------------------------------------------
# 3. 메타데이터(품목명, 카테고리) 병합 및 저장
# ----------------------------------------------------
meta_info = df[['hs6', 'hs6_name_kr', 'cbam_category']].drop_duplicates()

final_lr = pd.merge(df_pred_lr, meta_info, on='hs6', how='left')
final_es = pd.merge(df_pred_es, meta_info, on='hs6', how='left')

# 시각화를 위해 보기 편하도록 컬럼 순서 재배치
col_order = ['hs6', 'hs6_name_kr', 'cbam_category', 'Risk_Score', 
             'c1_scaled_pred', 'c2_scaled_pred', 'c3_scaled_pred']
final_lr = final_lr[col_order]
final_es = final_es[col_order]

# CSV로 각각 저장
final_lr.to_csv('../csv/processed/Risk_Score_LR_2026.csv', index=False, encoding='utf-8-sig')
final_es.to_csv('../csv/processed/Risk_Score_ES_2026.csv', index=False, encoding='utf-8-sig')

print("\n✅ 모델별 예측 및 스코어 산출 완료!")
print(f" 1. 선형 회귀(Linear Regression) 결과 저장: '../csv/processed/Risk_Score_LR_2026.csv'")
print(f" 2. 지수평활법(Exponential Smoothing) 결과 저장: '../csv/processed/Risk_Score_ES_2026.csv'\n")

print("📊 [미리보기] 선형 회귀 모델 기반 Top 5 위험 품목:")
print(final_lr.head())

데이터를 불러오는 중입니다...
[2026년 리스크 지표 예측 중 (선형회귀 & 지수평활법)]...

✅ 모델별 예측 및 스코어 산출 완료!
 1. 선형 회귀(Linear Regression) 결과 저장: '../csv/processed/Risk_Score_LR_2026.csv'
 2. 지수평활법(Exponential Smoothing) 결과 저장: '../csv/processed/Risk_Score_ES_2026.csv'

📊 [미리보기] 선형 회귀 모델 기반 Top 5 위험 품목:
      hs6                   hs6_name_kr cbam_category  Risk_Score  \
0  720712                           슬래브            철강    2.700772   
1  721934  스테인리스강 냉연(폭≥600mm 두께0.5mm미만)            철강    2.439549   
2  722519                  합금강 평판열연(기타)            철강    2.270752   
3  721935         스테인리스강 냉연(폭≥600mm 기타)            철강    2.226398   
4  721899                     스테인리스강 블룸            철강    2.220039   

   c1_scaled_pred  c2_scaled_pred  c3_scaled_pred  
0        5.200000        1.888998       -0.151947  
1        4.520386        2.136588       -0.463736  
2        5.134759        1.091513       -0.645726  
3        3.410103        2.749036       -0.409418  
4        2.971342        3.678228       -1.048264  

In [ ]:
import pandas as pd
import numpy as np

# 1. 데이터 불러오기 (지수평활법 결과)
print("데이터를 불러오고 등급을 부여합니다...")
df = pd.read_csv('../csv/processed/Risk_Score_ES_2026.csv')

# ==========================================
# [STEP 1] 경험적 임계값 기반 등급 부여
# ==========================================
# 상위 20% = RED, 상위 20~50% = YELLOW, 하위 50% = GREEN
thresh_red    = df['Risk_Score'].quantile(0.80)
thresh_yellow = df['Risk_Score'].quantile(0.50)

print(f"  RED  임계값 (80th percentile): {thresh_red:.6f}")
print(f"  YELLOW 임계값 (50th percentile): {thresh_yellow:.6f}")

df['Risk_Grade'] = df['Risk_Score'].apply(
    lambda s: 'RED' if s >= thresh_red else ('YELLOW' if s >= thresh_yellow else 'GREEN')
)

grade_counts = df['Risk_Grade'].value_counts()
print(f"\n  등급 분포: RED {grade_counts.get('RED',0)}개 / "
      f"YELLOW {grade_counts.get('YELLOW',0)}개 / "
      f"GREEN {grade_counts.get('GREEN',0)}개")

# ==========================================
# [STEP 2] 요인 기여도 분해
# ==========================================
w_c1, w_c2, w_c3 = 0.3986, 0.3525, 0.2489

df['Contribution_C1'] = df['c1_scaled_pred'] * w_c1
df['Contribution_C2'] = df['c2_scaled_pred'] * w_c2
df['Contribution_C3'] = df['c3_scaled_pred'] * w_c3

def get_main_factor(row):
    factors = {
        '잠재 부담액(C1)': row['Contribution_C1'],
        '수출 의존도(C2)': row['Contribution_C2'],
        '탄소 격차(C3)':   row['Contribution_C3'],
    }
    return max(factors, key=factors.get)

df['Main_Risk_Factor'] = df.apply(get_main_factor, axis=1)

# ==========================================
# [STEP 3] 결과 정리 및 저장
# ==========================================
columns_order = [
    'hs6', 'hs6_name_kr', 'cbam_category',
    'Risk_Score', 'Risk_Grade', 'Main_Risk_Factor',
    'Contribution_C1', 'Contribution_C2', 'Contribution_C3',
]
df_final = df[columns_order].sort_values(by='Risk_Score', ascending=False).reset_index(drop=True)

df_final.to_csv('../csv/processed/CBAM_Final_Risk_Analysis_2026.csv', index=False, encoding='utf-8-sig')

print("\n✅ 등급 부여 및 요인 분해 완료!")
print("저장: '../csv/processed/CBAM_Final_Risk_Analysis_2026.csv'\n")
print("📊 [미리보기] 상위 5개 품목:")
print(df_final[['hs6_name_kr', 'Risk_Score', 'Risk_Grade', 'Main_Risk_Factor']].head())
